# YOLOv8n-cls Hyperparameter Search and Training

이 노트북은 팀 기준 모델인 `yolov8n-cls.pt`를 사용해 OPEN/CLOSE 눈 상태 분류 모델을 학습합니다.

핵심 절차:
- Google Drive의 `eye_data_set.zip` 사용
- Optuna 기반 하이퍼파라미터 탐색
- validation 기준 우선순위별 best 모델 저장
- best F1 설정으로 최종 학습 및 test 평가

주의: 현재 데이터는 bbox 라벨이 없는 분류 데이터이므로 YOLO detection이 아니라 YOLO classification입니다.

## 1. Environment Setup

Ultralytics YOLO와 Optuna를 설치하고 GPU를 확인합니다.

In [ ]:
!pip -q install ultralytics optuna scikit-learn seaborn onnx onnxruntime

import sys
import torch
import ultralytics

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Ultralytics:', ultralytics.__version__)
if torch.cuda.is_available():
    !nvidia-smi

## 2. Mount Drive and Locate Dataset

Drive에서 `eye_data_set.zip`을 찾고, 출력 폴더를 `src/outputs/yolov8n_cls_optuna`로 설정합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil
import subprocess

DATA_ZIP_NAME = 'eye_data_set.zip'
PROJECT_FOLDER_CANDIDATES = ['Face_Detection', 'Face Detection', 'FaceDetection']

mydrive = Path('/content/drive/MyDrive')
project_dir = None
for name in PROJECT_FOLDER_CANDIDATES:
    p = mydrive / name
    if p.exists():
        project_dir = p
        break

zip_matches = list(mydrive.glob(f'**/{DATA_ZIP_NAME}'))
if not zip_matches:
    raise FileNotFoundError(f'{DATA_ZIP_NAME} not found under Google Drive MyDrive.')

data_zip = zip_matches[0]
if project_dir is None:
    project_dir = data_zip.parent

src_dir = project_dir / 'src'
src_dir.mkdir(parents=True, exist_ok=True)

output_dir = src_dir / 'outputs' / 'yolov8n_cls_optuna'
output_dir.mkdir(parents=True, exist_ok=True)

work_dir = Path('/content/yolov8n_eye_state')
extract_dir = work_dir / 'extracted'
work_dir.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', project_dir)
print('SRC_DIR:', src_dir)
print('DATA_ZIP:', data_zip)
print('OUTPUT_DIR:', output_dir)

## 3. Extract Dataset and Validate Structure

YOLO classification 학습은 `train/val/test/class_name` 폴더 구조를 그대로 사용합니다.

In [ ]:
SPLITS = ['train', 'val', 'test']
CLASSES = ['awake', 'sleepy']
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def is_dataset_root(p: Path) -> bool:
    return all((p / split / cls).exists() for split in SPLITS for cls in CLASSES)

def find_dataset_root(root: Path):
    if is_dataset_root(root):
        return root
    for candidate in root.rglob('*'):
        if candidate.is_dir() and is_dataset_root(candidate):
            return candidate
    return None

if not find_dataset_root(extract_dir):
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    print('Extracting dataset...')
    subprocess.run(['unzip', '-q', '-o', str(data_zip), '-d', str(extract_dir)], check=True)

data_dir = find_dataset_root(extract_dir)
if data_dir is None:
    raise FileNotFoundError('Could not find train/val/test awake/sleepy folders after extraction.')

counts = {}
for split in SPLITS:
    counts[split] = {}
    for cls in CLASSES:
        folder = data_dir / split / cls
        counts[split][cls] = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS)

print('DATA_DIR:', data_dir)
counts

## 4. Search Configuration

탐색 예산은 10 trials, trial당 5 epochs입니다. 앞 5 trials는 random startup 탐색, 이후 5 trials는 TPE 기반 Bayesian 탐색으로 진행합니다. 모든 trial은 `yolov8n-cls.pt`에서 시작합니다.

In [ ]:
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from ultralytics import YOLO
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

SEED = 42
N_TRIALS = 10
N_STARTUP_TRIALS = 5
TRIAL_EPOCHS = 5
FINAL_EPOCHS = 30
POSITIVE_CLASS_NAME = 'sleepy'  # sleepy 폴더를 CLOSE 클래스로 해석합니다.

random.seed(SEED)
np.random.seed(SEED)

runs_dir = output_dir / 'runs'
runs_dir.mkdir(parents=True, exist_ok=True)

## 5. Evaluation Helpers

Ultralytics validation 결과와 별도로, 같은 방식으로 CLOSE 기준 precision/recall/F1/PR-AUC를 계산합니다.

In [ ]:
def collect_image_paths(split: str):
    paths, labels = [], []
    class_to_idx = {cls: i for i, cls in enumerate(sorted(CLASSES))}
    for cls in sorted(CLASSES):
        for p in sorted((data_dir / split / cls).iterdir()):
            if p.is_file() and p.suffix.lower() in VALID_EXTS:
                paths.append(str(p))
                labels.append(class_to_idx[cls])
    return paths, np.array(labels), class_to_idx


def calc_metrics(y_true, probs, positive_idx):
    y_pred = probs.argmax(axis=1)
    pos_prob = probs[:, positive_idx]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[positive_idx], average='binary', pos_label=positive_idx, zero_division=0
    )
    out = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'close_precision': float(precision),
        'close_recall': float(recall),
        'f1': float(f1),
        'pr_auc': float(average_precision_score((np.array(y_true) == positive_idx).astype(int), pos_prob)),
    }
    try:
        out['roc_auc'] = float(roc_auc_score((np.array(y_true) == positive_idx).astype(int), pos_prob))
    except ValueError:
        out['roc_auc'] = float('nan')
    return out


def predict_probs_yolo(model_path, split: str, imgsz: int, batch: int):
    model = YOLO(str(model_path), task='classify')
    paths, y_true, class_to_idx = collect_image_paths(split)
    all_probs = []
    for i in range(0, len(paths), batch):
        chunk = paths[i:i + batch]
        results = model(chunk, imgsz=imgsz, device=0, verbose=False)
        for r in results:
            all_probs.append(r.probs.data.cpu().numpy())
    probs = np.vstack(all_probs)
    return y_true, probs, class_to_idx


def save_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')

## 6. Optuna Objective

각 trial은 YOLOv8n-cls를 짧게 학습하고 validation 성능을 계산합니다. 우선순위별 최고 모델은 별도 폴더에 저장합니다.

In [ ]:
priority_specs = {
    'best_by_close_recall': 'close_recall',
    'best_by_f1': 'f1',
    'best_by_pr_auc': 'pr_auc',
    'best_by_accuracy': 'accuracy',
}
best_scores = {name: -1.0 for name in priority_specs}
trial_rows = []


def save_priority_model(priority_name, best_pt, config, metrics):
    target = output_dir / priority_name
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best_pt, target / 'model.pt')
    save_json(target / 'config.json', config)
    save_json(target / 'metrics.json', metrics)


def objective(trial):
    imgsz = trial.suggest_categorical('imgsz', [96, 128, 160])
    batch = trial.suggest_categorical('batch', [64, 128])
    lr0 = trial.suggest_float('lr0', 1e-4, 1e-2, log=True)
    lrf = trial.suggest_float('lrf', 0.01, 0.2)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    degrees = trial.suggest_float('degrees', 0.0, 10.0)

    config = {
        'model': 'yolov8n-cls.pt',
        'imgsz': imgsz,
        'batch': batch,
        'lr0': lr0,
        'lrf': lrf,
        'weight_decay': weight_decay,
        'degrees': degrees,
        'trial_epochs': TRIAL_EPOCHS,
    }

    run_name = f'trial_{trial.number:03d}'
    model = YOLO('yolov8n-cls.pt')
    model.train(
        data=str(data_dir),
        epochs=TRIAL_EPOCHS,
        imgsz=imgsz,
        batch=batch,
        lr0=lr0,
        lrf=lrf,
        weight_decay=weight_decay,
        degrees=degrees,
        device=0,
        project=str(runs_dir),
        name=run_name,
        exist_ok=True,
        patience=2,
        verbose=False,
    )

    best_pt = runs_dir / run_name / 'weights' / 'best.pt'
    if not best_pt.exists():
        raise FileNotFoundError(f'Missing YOLO best.pt: {best_pt}')

    y_true, probs, class_to_idx = predict_probs_yolo(best_pt, 'val', imgsz, batch)
    positive_idx = class_to_idx[POSITIVE_CLASS_NAME]
    metrics = calc_metrics(y_true, probs, positive_idx)
    metrics.update({'trial': trial.number})

    row = {**config, **metrics, 'trial': trial.number}
    trial_rows.append(row)
    pd.DataFrame(trial_rows).to_csv(output_dir / 'tuning_results.csv', index=False)

    for priority_name, metric_name in priority_specs.items():
        score = metrics[metric_name]
        if score > best_scores[priority_name]:
            best_scores[priority_name] = score
            save_priority_model(priority_name, best_pt, {**config, 'trial': trial.number}, metrics)

    return metrics['f1']

## 7. Run Search

Optuna TPE sampler를 사용하되, 앞 5 trials는 random startup으로 두고 이후 5 trials에서 이전 결과를 반영해 YOLOv8n-cls의 주요 학습 파라미터를 탐색합니다.

In [ ]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=N_STARTUP_TRIALS),
    study_name='yolov8n_cls_eye_state',
    storage=f"sqlite:///{output_dir / 'optuna_study.db'}",
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS)

print('Best trial:', study.best_trial.number)
print('Best value:', study.best_value)
print('Best params:', study.best_params)

## 8. Final Training with Best F1 Config

탐색에서 validation F1-score가 가장 좋았던 설정으로 YOLOv8n-cls를 최종 학습합니다.

In [ ]:
best_params = study.best_params
final_config = {
    'model': 'yolov8n-cls.pt',
    'imgsz': best_params['imgsz'],
    'batch': best_params['batch'],
    'lr0': best_params['lr0'],
    'lrf': best_params['lrf'],
    'weight_decay': best_params['weight_decay'],
    'degrees': best_params['degrees'],
    'final_epochs': FINAL_EPOCHS,
}

final_dir = output_dir / 'final_best_by_f1'
final_dir.mkdir(parents=True, exist_ok=True)

model = YOLO('yolov8n-cls.pt')
model.train(
    data=str(data_dir),
    epochs=FINAL_EPOCHS,
    imgsz=final_config['imgsz'],
    batch=final_config['batch'],
    lr0=final_config['lr0'],
    lrf=final_config['lrf'],
    weight_decay=final_config['weight_decay'],
    degrees=final_config['degrees'],
    device=0,
    project=str(output_dir),
    name='final_best_by_f1_run',
    exist_ok=True,
    patience=8,
)

best_pt = output_dir / 'final_best_by_f1_run' / 'weights' / 'best.pt'
shutil.copy2(best_pt, final_dir / 'model.pt')
save_json(final_dir / 'config.json', final_config)
print('Saved final model:', final_dir / 'model.pt')

## 9. Test Evaluation and ONNX Export

최종 모델은 test set에서 한 번만 평가하고 ONNX로 export합니다.

In [ ]:
model_pt = final_dir / 'model.pt'
y_true, probs, class_to_idx = predict_probs_yolo(model_pt, 'test', final_config['imgsz'], final_config['batch'])
positive_idx = class_to_idx[POSITIVE_CLASS_NAME]
test_metrics = calc_metrics(y_true, probs, positive_idx)
y_pred = probs.argmax(axis=1)
cm = confusion_matrix(y_true, y_pred)

save_json(final_dir / 'test_metrics.json', test_metrics)
print('Test metrics:', test_metrics)

plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=sorted(CLASSES), yticklabels=sorted(CLASSES), cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('YOLOv8n-cls Confusion Matrix')
plt.tight_layout()
plt.savefig(final_dir / 'confusion_matrix.png', dpi=150)
plt.show()

model = YOLO(str(model_pt), task='classify')
exported = model.export(format='onnx', imgsz=final_config['imgsz'], opset=17, simplify=True)
shutil.copy2(exported, final_dir / 'model.onnx')
print('Saved:', final_dir)